In [2]:

import pandas as pd

balls = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\ball_by_ball.csv")
matches = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\matches.csv")

# Calculate total runs
balls["total_runs"] = (
    balls["runs"] + balls["extras1_runs"] + balls["extras_2_runs"]
)

# Identify legal deliveries
balls["legal_ball"] = ~balls["extras_type"].isin(["wide", "Overstep"])

# Add match information
balls = balls.merge(
    matches[["match_id", "revised_target",
             "first_inn_revised_overs", "second_inn_revised_overs"]],
    on="match_id", how="left"
)

# Sort deliveries
balls = balls.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).copy()

# Calculate runs and wickets before each delivery
group = balls.groupby(["match_id", "innings"])

balls["runs_before"] = group["total_runs"].cumsum() - balls["total_runs"]
balls["wickets_before"] = group["is_wicket"].cumsum() - balls["is_wicket"]

# Calculate legal balls before each delivery
balls["legal_before"] = (
    group["legal_ball"].cumsum() - balls["legal_ball"].astype(int)
)

# Calculate remaining balls
balls["max_overs"] = 20

balls.loc[
    balls["innings"] == 1, "max_overs"
] = balls["first_inn_revised_overs"].fillna(20)

balls.loc[
    balls["innings"] == 2, "max_overs"
] = balls["second_inn_revised_overs"].fillna(20)

balls["balls_remaining"] = (
    balls["max_overs"] * 6 - balls["legal_before"]
).clip(lower=0)

# Get first innings total for each match
first_total = (
    balls[balls["innings"] == 1]
    .groupby("match_id")["total_runs"].sum()
    .add(1)
)

balls["target"] = balls["match_id"].map(first_total)

# Use revised target when available
balls["target"] = balls["revised_target"].fillna(balls["target"])

# Calculate required run rate
balls["runs_needed"] = (
    balls["target"] - balls["runs_before"]
).clip(lower=0)

balls["required_rate"] = (
    balls["runs_needed"] * 6 /
    balls["balls_remaining"].replace(0, float("nan"))
)

# Pressure indicators
balls["death_over"] = balls["overid"] >= 16
balls["low_wickets"] = (10 - balls["wickets_before"]) <= 4

balls["high_required_rate"] = (
    (balls["innings"] == 2) &
    (balls["required_rate"] >= 10)
)

balls["pressure_score"] = (
    balls["death_over"].astype(int) +
    balls["low_wickets"].astype(int) +
    balls["high_required_rate"].astype(int)
)

balls["pressure"] = balls["pressure_score"] >= 2

balls.to_csv("Results/pressure_deliveries.csv", index=False)

print("Total deliveries:", len(balls))
print("Pressure deliveries:", balls["pressure"].sum())
print("Normal deliveries:", (~balls["pressure"]).sum())
print("Saved pressure_deliveries.csv")


Total deliveries: 200999
Pressure deliveries: 25326
Normal deliveries: 175673
Saved pressure_deliveries.csv


In [3]:

import pandas as pd

balls = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\ball_by_ball.csv")
matches = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\matches.csv")

print("Filtered ball records:", len(balls))
print("Filtered match records:", len(matches))

print("Unique ball match IDs:", balls["match_id"].nunique())
print("Unique match IDs:", matches["match_id"].nunique())

print("Duplicate match IDs:", matches["match_id"].duplicated().sum())


Filtered ball records: 200999
Filtered match records: 883
Unique ball match IDs: 867
Unique match IDs: 883
Duplicate match IDs: 0


In [4]:

import pandas as pd
import os

balls = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\ball_by_ball.csv")
matches = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\matches.csv")
comps = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\competitions.csv")

# Select IPL and T20I
selected = comps[
    comps["comp_name"].str.contains("IPL", case=False, na=False) |
    (comps["match_format"] == "T20I")
]

# Filter matches
filtered_matches = matches[
    matches["comp_id"].isin(selected["comp_id"])
].copy()

# Filter deliveries
filtered_balls = balls[
    balls["match_id"].isin(filtered_matches["match_id"])
].copy()

os.makedirs("Results", exist_ok=True)

filtered_matches.to_csv("Results/ipl_t20i_matches.csv", index=False)
filtered_balls.to_csv("Results/ipl_t20i_balls.csv", index=False)

print("Filtered matches:", len(filtered_matches))
print("Filtered deliveries:", len(filtered_balls))
print("Unique match IDs:", filtered_balls["match_id"].nunique())


Filtered matches: 853
Filtered deliveries: 194014
Unique match IDs: 837


In [7]:
%run Pressure_analysis.ipynb

Total deliveries: 200999
Pressure deliveries: 25326
Normal deliveries: 175673
Saved pressure_deliveries.csv
Filtered ball records: 200999
Filtered match records: 883
Unique ball match IDs: 867
Unique match IDs: 883
Duplicate match IDs: 0
Filtered matches: 853
Filtered deliveries: 194014
Unique match IDs: 837


In [ ]:

import pandas as pd

balls = pd.read_csv(r"Results/ipl_t20i_balls.csv")

print("Total deliveries:", len(balls))
print("Unique matches:", balls["match_id"].nunique())
print("Dataset loaded correctly!")


In [ ]:
%run Pressure_analysis.ipynb

In [1]:
print("Jupyter is working!")

Jupyter is working!


In [3]:

import pandas as pd

balls = pd.read_csv(r"Results/ipl_t20i_balls.csv")

# Calculate wickets before each ball
balls = balls.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).copy()

group = balls.groupby(["match_id", "innings"])

balls["wickets_before"] = (
    group["is_wicket"].cumsum() - balls["is_wicket"]
)

# Calculate remaining wickets
balls["wickets_left"] = 10 - balls["wickets_before"]

# Pressure indicators
balls["death_over"] = balls["overid"] >= 16
balls["low_wickets"] = balls["wickets_left"] <= 4

# Calculate pressure score
balls["pressure_score"] = (
    balls["death_over"].astype(int) +
    balls["low_wickets"].astype(int)
)

# Identify pressure deliveries
balls["pressure"] = balls["pressure_score"] >= 1

print("Total deliveries:", len(balls))
print("Pressure deliveries:", balls["pressure"].sum())
print("Normal deliveries:", (~balls["pressure"]).sum())

balls.to_csv("Results/pressure_deliveries.csv", index=False)


Total deliveries: 194014
Pressure deliveries: 48751
Normal deliveries: 145263


In [5]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# Calculate batting performance
batting = balls.groupby(
    ["striker_id", "striker_name", "pressure"]
).agg(
    runs=("runs", "sum"),
    balls=("extras_type", lambda x: (x != "wide").sum()),
    fours=("is_four", "sum"),
    sixes=("is_six", "sum"),
    matches=("match_id", "nunique")
).reset_index()

# Calculate strike rate
batting["strike_rate"] = (
    batting["runs"] / batting["balls"] * 100
).round(2)

# Separate pressure and normal performance
pressure_bat = batting[batting["pressure"] == True]
normal_bat = batting[batting["pressure"] == False]

result = pressure_bat.merge(
    normal_bat,
    on=["striker_id", "striker_name"],
    suffixes=("_pressure", "_normal")
)

result["strike_rate_difference"] = (
    result["strike_rate_pressure"] -
    result["strike_rate_normal"]
).round(2)

result = result.sort_values(
    "strike_rate_pressure", ascending=False
)

result.to_csv("Results/batter_pressure_comparison.csv", index=False)

print("===== TOP 15 BATTERS UNDER PRESSURE =====")
print(result[
    ["striker_name", "runs_pressure",
     "balls_pressure", "strike_rate_pressure",
     "strike_rate_normal", "strike_rate_difference"]
].head(15).to_string(index=False))


===== TOP 15 BATTERS UNDER PRESSURE =====
        striker_name  runs_pressure  balls_pressure  strike_rate_pressure  strike_rate_normal  strike_rate_difference
      Ajinkya Rahane             34              10                340.00              137.14                  202.86
 Dhananjaya de Silva             17               5                340.00              118.04                  221.96
           Azam Khan             46              15                306.67               83.72                  222.95
Vaibhav Sooryavanshi             29              10                290.00              218.49                   71.51
     Vikramjit Singh             27              10                270.00              127.56                  142.44
         Keacy Carty             35              13                269.23               76.27                  192.96
          Haider Ali             18               7                257.14              108.96                  148.18
 Ravichandran 

In [6]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# Identify batter dismissals
balls["batter_out"] = (
    (balls["is_wicket"] == 1) &
    (balls["dismissed_player_id"] == balls["striker_id"])
)

# Calculate batting statistics
batting = balls.groupby(
    ["striker_id", "striker_name", "pressure"]
).agg(
    runs=("runs", "sum"),
    balls=("extras_type", lambda x: (x != "wide").sum()),
    dismissals=("batter_out", "sum"),
    matches=("match_id", "nunique")
).reset_index()

batting["strike_rate"] = (
    batting["runs"] / batting["balls"] * 100
)

batting["average"] = (
    batting["runs"] / batting["dismissals"].replace(0, float("nan"))
)

# Separate pressure and normal performance
p = batting[batting["pressure"] == True]
n = batting[batting["pressure"] == False]

result = p.merge(
    n,
    on=["striker_id", "striker_name"],
    suffixes=("_pressure", "_normal")
)

result["sr_difference"] = (
    result["strike_rate_pressure"] -
    result["strike_rate_normal"]
)

# Minimum experience requirements
result = result[
    (result["balls_pressure"] >= 100) &
    (result["matches_pressure"] >= 10)
]

result = result.sort_values(
    "strike_rate_pressure", ascending=False
)

result.to_csv("Results/reliable_batters.csv", index=False)

print("===== RELIABLE BATTERS UNDER PRESSURE =====")
print(result[
    ["striker_name", "runs_pressure",
     "balls_pressure", "matches_pressure",
     "average_pressure", "strike_rate_pressure",
     "strike_rate_normal", "sr_difference"]
].head(15).round(2).to_string(index=False))


===== RELIABLE BATTERS UNDER PRESSURE =====
    striker_name  runs_pressure  balls_pressure  matches_pressure  average_pressure  strike_rate_pressure  strike_rate_normal  sr_difference
Suryakumar Yadav            832             385                45             29.71                216.10              158.38          57.73
   Rajat Patidar            289             136                19             18.06                212.50              160.10          52.40
   Sai Sudharsan            308             149                19             22.00                206.71              143.59          63.12
Liam Livingstone            574             280                31             26.09                205.00              133.89          71.11
   Glenn Maxwell            356             177                18             35.60                201.13              151.86          49.27
      Naman Dhir            420             209                23             30.00                200.96     

In [7]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# Correct balls faced
balls["balls_faced"] = ~balls["extras_type"].isin(
    ["wide", "Overstep"]
)

# Identify batter dismissals
balls["batter_out"] = (
    (balls["is_wicket"] == 1) &
    (balls["dismissed_player_id"] == balls["striker_id"])
)

# Calculate statistics
batting = balls.groupby(
    ["striker_id", "striker_name", "pressure"]
).agg(
    runs=("runs", "sum"),
    balls=("balls_faced", "sum"),
    dismissals=("batter_out", "sum"),
    matches=("match_id", "nunique")
).reset_index()

batting["strike_rate"] = (
    batting["runs"] / batting["balls"] * 100
)

batting["average"] = (
    batting["runs"] / batting["dismissals"].replace(0, float("nan"))
)

# Separate pressure and normal statistics
p = batting[batting["pressure"] == True]
n = batting[batting["pressure"] == False]

result = p.merge(
    n,
    on=["striker_id", "striker_name"],
    suffixes=("_pressure", "_normal")
)

# Minimum experience
result = result[
    (result["balls_pressure"] >= 100) &
    (result["matches_pressure"] >= 10) &
    (result["balls_normal"] > 0) &
    (result["dismissals_pressure"] > 0)
].copy()

# Compare performance
result["sr_improvement"] = (
    result["strike_rate_pressure"] -
    result["strike_rate_normal"]
)

# Convert metrics into percentile scores
result["sr_score"] = result["strike_rate_pressure"].rank(pct=True)
result["avg_score"] = result["average_pressure"].rank(pct=True)
result["improvement_score"] = result["sr_improvement"].rank(pct=True)
result["runs_score"] = result["runs_pressure"].rank(pct=True)

# Final rating
result["rating"] = (
    result["sr_score"] * 30 +
    result["avg_score"] * 30 +
    result["improvement_score"] * 25 +
    result["runs_score"] * 15
).round(2)

result = result.sort_values("rating", ascending=False)

result.to_csv("Results/final_batter_ratings.csv", index=False)

print("===== TOP 10 BATTER RATINGS =====")
print(result[
    ["striker_name", "runs_pressure",
     "balls_pressure", "matches_pressure",
     "strike_rate_pressure", "average_pressure",
     "sr_improvement", "rating"]
].head(10).round(2).to_string(index=False))


===== TOP 10 BATTER RATINGS =====
     striker_name  runs_pressure  balls_pressure  matches_pressure  strike_rate_pressure  average_pressure  sr_improvement  rating
      Rinku Singh           1179             622                61                189.55             40.66           70.19   90.47
     Rishabh Pant            498             248                26                200.81             33.20           71.92   87.56
 Suryakumar Yadav            832             383                45                217.23             29.71           58.39   85.85
   Tristan Stubbs            762             403                41                189.08             40.11           62.05   85.68
  Ashutosh Sharma            493             254                24                194.09             29.00          100.67   84.57
Najibullah Zadran            228             117                11                194.87             45.60           90.06   84.42
 Liam Livingstone            574             279 

In [8]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# Legal balls for bowling
balls["legal_ball"] = ~balls["extras_type"].isin(
    ["wide", "Overstep"]
)

# Runs conceded by bowler
balls["runs_conceded"] = (
    balls["runs"] + balls["extras1_runs"] + balls["extras_2_runs"]
)

# Byes and legbyes are not charged to the bowler
balls.loc[
    balls["extras_type"].isin(["byes", "legbyes", "Penalty"]),
    "runs_conceded"
] = balls["runs"]

# Wickets credited to the bowler
credited = [
    "Caught", "Bowled", "LBW",
    "Stump", "Hit Wicket"
]

balls["bowler_wicket"] = (
    (balls["is_wicket"] == 1) &
    (balls["dismissal_type"].isin(credited))
)

# Calculate bowling statistics
bowling = balls.groupby(
    ["bowler_id", "bowler_name", "pressure"]
).agg(
    runs=("runs_conceded", "sum"),
    balls=("legal_ball", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

# Economy and bowling average
bowling["economy"] = (
    bowling["runs"] * 6 /
    bowling["balls"].replace(0, float("nan"))
)

bowling["average"] = (
    bowling["runs"] /
    bowling["wickets"].replace(0, float("nan"))
)

# Separate pressure and normal performance
p = bowling[bowling["pressure"] == True]
n = bowling[bowling["pressure"] == False]

result = p.merge(
    n,
    on=["bowler_id", "bowler_name"],
    suffixes=("_pressure", "_normal")
)

# Minimum experience
result = result[
    (result["balls_pressure"] >= 60) &
    (result["matches_pressure"] >= 10) &
    (result["wickets_pressure"] >= 1) &
    (result["balls_normal"] > 0)
].copy()

# Economy improvement: positive means better under pressure
result["economy_improvement"] = (
    result["economy_normal"] -
    result["economy_pressure"]
)

result = result.sort_values(
    "economy_pressure", ascending=True
)

result.to_csv("Results/bowler_pressure_comparison.csv", index=False)

print("===== TOP 15 BOWLERS UNDER PRESSURE =====")
print(result[
    ["bowler_name", "runs_pressure",
     "balls_pressure", "wickets_pressure",
     "matches_pressure", "economy_pressure",
     "average_pressure", "economy_normal",
     "economy_improvement"]
].head(15).round(2).to_string(index=False))



===== TOP 15 BOWLERS UNDER PRESSURE =====
     bowler_name  runs_pressure  balls_pressure  wickets_pressure  matches_pressure  economy_pressure  average_pressure  economy_normal  economy_improvement
     Abrar Ahmed            174             205                15                25              5.09             11.60            7.27                 2.17
  Mohammad Nawaz            105             120                 9                15              5.25             11.67            7.38                 2.13
  Sufiyan Muqeem            101             106                12                12              5.72              8.42            6.83                 1.11
     Shadab Khan            128             129                12                17              5.95             10.67            8.27                 2.32
    Sunil Narine            291             286                23                43              6.10             12.65            7.04                 0.93
   Mohammad Nabi

In [9]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# Legal deliveries
balls["legal_ball"] = ~balls["extras_type"].isin(
    ["wide", "Overstep"]
)

# Bowler runs conceded
balls["runs_conceded"] = (
    balls["runs"] + balls["extras1_runs"] + balls["extras_2_runs"]
)

balls.loc[
    balls["extras_type"].isin(["byes", "legbyes", "Penalty"]),
    "runs_conceded"
] = balls["runs"]

# Bowler wickets
credited = ["Caught", "Bowled", "LBW", "Stump", "Hit Wicket"]

balls["bowler_wicket"] = (
    (balls["is_wicket"] == 1) &
    balls["dismissal_type"].isin(credited)
)

# Group bowling performance
bowling = balls.groupby(
    ["bowler_id", "bowler_name", "pressure"]
).agg(
    runs=("runs_conceded", "sum"),
    balls=("legal_ball", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

bowling["economy"] = (
    bowling["runs"] * 6 / bowling["balls"].replace(0, float("nan"))
)

bowling["average"] = (
    bowling["runs"] / bowling["wickets"].replace(0, float("nan"))
)

bowling["wicket_rate"] = (
    bowling["wickets"] * 100 /
    bowling["balls"].replace(0, float("nan"))
)

# Separate pressure and normal performance
p = bowling[bowling["pressure"] == True]
n = bowling[bowling["pressure"] == False]

result = p.merge(
    n,
    on=["bowler_id", "bowler_name"],
    suffixes=("_pressure", "_normal")
)

# Minimum experience
result = result[
    (result["balls_pressure"] >= 60) &
    (result["matches_pressure"] >= 10) &
    (result["wickets_pressure"] >= 1) &
    (result["balls_normal"] > 0)
].copy()

# Economy improvement
result["economy_improvement"] = (
    result["economy_normal"] - result["economy_pressure"]
)

# Percentile scores (lower economy and average are better)
result["economy_score"] = result["economy_pressure"].rank(
    ascending=False, pct=True
)
result["average_score"] = result["average_pressure"].rank(
    ascending=False, pct=True
)
result["wicket_rate_score"] = result["wicket_rate_pressure"].rank(pct=True)
result["improvement_score"] = result["economy_improvement"].rank(pct=True)
result["wickets_score"] = result["wickets_pressure"].rank(pct=True)

# Final rating
result["rating"] = (
    result["economy_score"] * 30 +
    result["average_score"] * 25 +
    result["wicket_rate_score"] * 20 +
    result["improvement_score"] * 15 +
    result["wickets_score"] * 10
).round(2)

result = result.sort_values("rating", ascending=False)

result.to_csv("Results/final_bowler_ratings.csv", index=False)

print("===== TOP 10 BOWLER RATINGS =====")
print(result[
    ["bowler_name", "runs_pressure",
     "balls_pressure", "wickets_pressure",
     "matches_pressure", "economy_pressure",
     "average_pressure", "wicket_rate_pressure",
     "rating"]
].head(10).round(2).to_string(index=False))



===== TOP 10 BOWLER RATINGS =====
      bowler_name  runs_pressure  balls_pressure  wickets_pressure  matches_pressure  economy_pressure  average_pressure  wicket_rate_pressure  rating
    Kuldeep Yadav            478             390                38                56              7.35             12.58                  9.74   92.94
   Sufiyan Muqeem            101             106                12                12              5.72              8.42                 11.32   92.47
      Shadab Khan            128             129                12                17              5.95             10.67                  9.30   90.12
Wanindu Hasaranga            512             382                40                53              8.04             12.80                 10.47   89.53
     Akeal Hosein            163             131                13                20              7.47             12.54                  9.92   88.71
      Rashid Khan            915             688            

In [10]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# No-ball counts as a batter ball faced; wide does not
balls["batter_ball"] = balls["extras_type"] != "wide"

# Batter statistics
batting = balls.groupby(
    ["striker_id", "striker_name", "pressure"]
).agg(
    runs=("runs", "sum"),
    balls=("batter_ball", "sum"),
    matches=("match_id", "nunique"),
    dismissals=("dismissed_player_id",
                lambda x: x.notna().sum())
).reset_index()

# Strike rate and batting average
batting["strike_rate"] = (
    batting["runs"] * 100 /
    batting["balls"].replace(0, float("nan"))
)

batting["average"] = (
    batting["runs"] /
    batting["dismissals"].replace(0, float("nan"))
)

# Separate pressure and normal performance
p = batting[batting["pressure"] == True]
n = batting[batting["pressure"] == False]

result = p.merge(
    n,
    on=["striker_id", "striker_name"],
    suffixes=("_pressure", "_normal")
)

# Minimum experience
result = result[
    (result["balls_pressure"] >= 100) &
    (result["matches_pressure"] >= 10) &
    (result["dismissals_pressure"] > 0) &
    (result["balls_normal"] > 0)
].copy()

# Improvement in strike rate
result["sr_improvement"] = (
    result["strike_rate_pressure"] -
    result["strike_rate_normal"]
)

# Percentile scores
result["sr_score"] = result["strike_rate_pressure"].rank(pct=True)
result["average_score"] = result["average_pressure"].rank(pct=True)
result["improvement_score"] = result["sr_improvement"].rank(pct=True)
result["runs_score"] = result["runs_pressure"].rank(pct=True)

# Final batting rating
result["rating"] = (
    result["sr_score"] * 30 +
    result["average_score"] * 30 +
    result["improvement_score"] * 25 +
    result["runs_score"] * 15
).round(2)

result = result.sort_values("rating", ascending=False)

result.to_csv("Results/final_batter_ratings.csv", index=False)

print("===== TOP 10 BATTER RATINGS =====")
print(result[
    ["striker_name", "runs_pressure",
     "balls_pressure", "matches_pressure",
     "strike_rate_pressure", "average_pressure",
     "sr_improvement", "rating"]
].head(10).round(2).to_string(index=False))


===== TOP 10 BATTER RATINGS =====
        striker_name  runs_pressure  balls_pressure  matches_pressure  strike_rate_pressure  average_pressure  sr_improvement  rating
         Rinku Singh           1179             627                61                188.04             36.84           69.04   89.19
        Rishabh Pant            498             248                26                200.81             33.20           72.18   88.29
    Suryakumar Yadav            832             385                45                216.10             29.71           57.73   86.50
      Tristan Stubbs            762             405                41                188.15             38.10           61.12   85.73
   Najibullah Zadran            228             119                11                191.60             45.60           86.78   84.21
     Ashutosh Sharma            493             258                24                191.09             27.39           97.66   83.80
    Liam Livingstone        

In [11]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

print("Dismissal types:")
print(balls["dismissal_type"].value_counts(dropna=False))

print("\nOther extras:")
print(
    balls[balls["extras_type"] == "other"][
        ["runs", "extras1_runs", "extras_2_runs",
         "dismissal_type", "is_wicket"]
    ].head(20).to_string(index=False)
)


Dismissal types:
dismissal_type
NaN                      183528
Caught                     7332
Bowled                     1690
LBW                         666
RunOut                      573
Stump                       200
Hit Wicket                   17
Retire Out                    6
Obstracting the field         1
Obstructing the field         1
Name: count, dtype: int64

Other extras:
 runs  extras1_runs  extras_2_runs dismissal_type  is_wicket
    0             1              0            NaN      False
    0             1              0            NaN      False
    2             1              0            NaN      False
    0             1              4            NaN      False
    0             1              0            NaN      False
    6             1              0            NaN      False
    0             1              0            NaN      False
    0             1              0            NaN      False
    0             1              0         RunOut       Tr

In [12]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# Balls faced: wides are excluded; no-balls are included
balls["batter_ball"] = balls["extras_type"] != "wide"

# Count a dismissal only when the striker was dismissed
balls["batter_dismissal"] = (
    (balls["is_wicket"] == True) &
    (balls["dismissed_player_id"] == balls["striker_id"])
)

# Calculate batting statistics
batting = balls.groupby(
    ["striker_id", "striker_name", "pressure"]
).agg(
    runs=("runs", "sum"),
    balls=("batter_ball", "sum"),
    matches=("match_id", "nunique"),
    dismissals=("batter_dismissal", "sum")
).reset_index()

batting["strike_rate"] = (
    batting["runs"] * 100 /
    batting["balls"].replace(0, float("nan"))
)

batting["average"] = (
    batting["runs"] /
    batting["dismissals"].replace(0, float("nan"))
)

# Separate pressure and normal performance
p = batting[batting["pressure"] == True]
n = batting[batting["pressure"] == False]

result = p.merge(
    n,
    on=["striker_id", "striker_name"],
    suffixes=("_pressure", "_normal")
)

# Minimum experience
result = result[
    (result["balls_pressure"] >= 100) &
    (result["matches_pressure"] >= 10) &
    (result["dismissals_pressure"] > 0) &
    (result["balls_normal"] > 0)
].copy()

# Strike-rate improvement
result["sr_improvement"] = (
    result["strike_rate_pressure"] -
    result["strike_rate_normal"]
)

# Rating calculation
result["sr_score"] = result["strike_rate_pressure"].rank(pct=True)
result["average_score"] = result["average_pressure"].rank(pct=True)
result["improvement_score"] = result["sr_improvement"].rank(pct=True)
result["runs_score"] = result["runs_pressure"].rank(pct=True)

result["rating"] = (
    result["sr_score"] * 30 +
    result["average_score"] * 30 +
    result["improvement_score"] * 25 +
    result["runs_score"] * 15
).round(2)

result = result.sort_values("rating", ascending=False)

result.to_csv("Results/final_batter_ratings.csv", index=False)

print("===== CORRECTED TOP 10 BATTERS =====")
print(result[
    ["striker_name", "runs_pressure",
     "balls_pressure", "matches_pressure",
     "dismissals_pressure", "strike_rate_pressure",
     "average_pressure", "sr_improvement", "rating"]
].head(10).round(2).to_string(index=False))


===== CORRECTED TOP 10 BATTERS =====
        striker_name  runs_pressure  balls_pressure  matches_pressure  dismissals_pressure  strike_rate_pressure  average_pressure  sr_improvement  rating
         Rinku Singh           1179             627                61                   29                188.04             40.66           69.04   89.96
        Rishabh Pant            498             248                26                   15                200.81             33.20           72.18   87.78
    Suryakumar Yadav            832             385                45                   28                216.10             29.71           57.73   85.85
      Tristan Stubbs            762             405                41                   19                188.15             40.11           61.12   85.73
     Ashutosh Sharma            493             258                24                   17                191.09             29.00           97.66   84.06
   Najibullah Zadran            2

In [13]:

balls = pd.read_csv("Results/pressure_deliveries.csv")

other = balls[balls["extras_type"] == "other"]

print("Other extras count:", len(other))
print("Total runs:", other["runs"].sum())
print("Extra 1 runs:", other["extras1_runs"].sum())
print("Extra 2 runs:", other["extras_2_runs"].sum())

print("\nOther extras by dismissal:")
print(other["dismissal_type"].value_counts(dropna=False))


Other extras count: 260
Total runs: 411
Extra 1 runs: 260
Extra 2 runs: 34

Other extras by dismissal:
dismissal_type
NaN       258
RunOut      2
Name: count, dtype: int64


In [14]:

import pandas as pd

balls = pd.read_csv("Results/pressure_deliveries.csv")

# Legal balls exclude wides and no-balls
balls["legal_ball"] = ~balls["extras_type"].isin(
    ["wide", "Overstep"]
)

# Start with batter runs
balls["runs_conceded"] = balls["runs"]

# Add extras charged to bowler
charged = balls["extras_type"].isin(["wide", "Overstep"])

balls.loc[charged, "runs_conceded"] += (
    balls.loc[charged, "extras1_runs"] +
    balls.loc[charged, "extras_2_runs"]
)

# Bowler-credited wickets
credited = ["Caught", "Bowled", "LBW", "Stump", "Hit Wicket"]

balls["bowler_wicket"] = (
    (balls["is_wicket"] == True) &
    balls["dismissal_type"].isin(credited)
)

# Bowling statistics
bowling = balls.groupby(
    ["bowler_id", "bowler_name", "pressure"]
).agg(
    runs=("runs_conceded", "sum"),
    balls=("legal_ball", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

bowling["economy"] = (
    bowling["runs"] * 6 /
    bowling["balls"].replace(0, float("nan"))
)

bowling["average"] = (
    bowling["runs"] /
    bowling["wickets"].replace(0, float("nan"))
)

bowling["wicket_rate"] = (
    bowling["wickets"] * 100 /
    bowling["balls"].replace(0, float("nan"))
)

# Compare pressure and normal performance
p = bowling[bowling["pressure"] == True]
n = bowling[bowling["pressure"] == False]

result = p.merge(
    n,
    on=["bowler_id", "bowler_name"],
    suffixes=("_pressure", "_normal")
)

result = result[
    (result["balls_pressure"] >= 60) &
    (result["matches_pressure"] >= 10) &
    (result["wickets_pressure"] >= 1) &
    (result["balls_normal"] > 0)
].copy()

result["economy_improvement"] = (
    result["economy_normal"] -
    result["economy_pressure"]
)

# Percentile scoring
result["economy_score"] = result["economy_pressure"].rank(
    ascending=False, pct=True
)
result["average_score"] = result["average_pressure"].rank(
    ascending=False, pct=True
)
result["wicket_rate_score"] = result["wicket_rate_pressure"].rank(pct=True)
result["improvement_score"] = result["economy_improvement"].rank(pct=True)
result["wickets_score"] = result["wickets_pressure"].rank(pct=True)

# Final bowling rating
result["rating"] = (
    result["economy_score"] * 30 +
    result["average_score"] * 25 +
    result["wicket_rate_score"] * 20 +
    result["improvement_score"] * 15 +
    result["wickets_score"] * 10
).round(2)

result = result.sort_values("rating", ascending=False)

result.to_csv("Results/final_bowler_ratings.csv", index=False)

print("===== CORRECTED TOP 10 BOWLERS =====")
print(result[
    ["bowler_name", "runs_pressure",
     "balls_pressure", "wickets_pressure",
     "matches_pressure", "economy_pressure",
     "average_pressure", "rating"]
].head(10).round(2).to_string(index=False))


===== CORRECTED TOP 10 BOWLERS =====
      bowler_name  runs_pressure  balls_pressure  wickets_pressure  matches_pressure  economy_pressure  average_pressure  rating
    Kuldeep Yadav            478             390                38                56              7.35             12.58   92.94
   Sufiyan Muqeem            101             106                12                12              5.72              8.42   92.56
      Shadab Khan            128             129                12                17              5.95             10.67   90.12
Wanindu Hasaranga            512             382                40                53              8.04             12.80   89.53
     Akeal Hosein            163             131                13                20              7.47             12.54   88.71
      Rashid Khan            915             688                68                88              7.98             13.46   87.94
   Shaheen Afridi            396             297            

In [15]:
import os
print(os.getcwd())

C:\Users\Admin\RR Selecctor\main files(code)
